# CSE 590A — Post-Discharge Follow-Up Agent · Colab Runner

End-to-end GPU run of the systems study. Three configs:

| Config | What | Prefix cache |
|--------|------|--------------|
| **A** | single-shot baseline (one LLM call/patient) | n/a |
| **B** | multi-agent (3 workers + assembler + reflection) | **OFF** |
| **C** | multi-agent (identical code to B) | **ON** |

The only difference between B and C is a vLLM *server* flag, so the server is
relaunched between them. We run **A + B under the cache-OFF server**, then
restart with cache ON for **C**, then evaluate.

> **Runtime:** set *Runtime → Change runtime type → T4 GPU* before running.
> **Heads-up:** `results/` is git-ignored — download it (last section) before the
> Colab VM disconnects, or the numbers are lost.

## 0. Confirm the GPU (expect a T4)

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q vllm openai requests matplotlib numpy huggingface_hub

## 2. Clone the project

Pulls the latest `main`. Safe to re-run (it just fast-forwards).

In [ ]:
import os
if not os.path.isdir('cse590a-project'):
    !git clone https://github.com/sais14/cse590a-project.git
%cd cse590a-project
!git pull --ff-only

## 3. Hugging Face login (gated model)

`meta-llama/Llama-3.2-3B-Instruct` is gated. **Before running this cell:** open
the model page, request access, and accept the license. Then paste a token with
**read** scope below.

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with read access to meta-llama/Llama-3.2-3B-Instruct

## 4. Build the data pipeline

`setup.sh` downloads the Synthea sample, builds `data/synthea.db`, and generates
the discharge notes. Idempotent — skips work that's already done.

In [ ]:
!bash setup.sh

## 5. vLLM server helpers

Launch vLLM in the background, poll `/health` until ready, run benchmarks, then
stop it cleanly before relaunching with a different cache setting.

In [ ]:
import subprocess, time, requests, signal

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
SERVER_LOG = "/content/vllm.log"
_proc = None

def launch_vllm(prefix_cache: bool):
    global _proc
    cache_flag = "--enable-prefix-caching" if prefix_cache else "--no-enable-prefix-caching"
    cmd = [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--dtype", "float16",
        "--max-model-len", "8192",
        "--port", "8000",
        "--gpu-memory-utilization", "0.9",
        cache_flag,
    ]
    log = open(SERVER_LOG, "w")
    _proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    print(f"launched vLLM (prefix_cache={prefix_cache}), pid={_proc.pid}; logging -> {SERVER_LOG}")
    return _proc

def wait_for_server(timeout=900):
    start = time.time()
    url = "http://localhost:8000/health"
    while time.time() - start < timeout:
        if _proc is not None and _proc.poll() is not None:
            raise RuntimeError(f"vLLM exited early (code {_proc.returncode}); see {SERVER_LOG}")
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print(f"server ready in {time.time()-start:.0f}s")
                return
        except Exception:
            pass
        time.sleep(3)
    raise TimeoutError(f"server not ready after {timeout}s; see {SERVER_LOG}")

def stop_vllm():
    global _proc
    if _proc is None:
        return
    _proc.send_signal(signal.SIGINT)
    try:
        _proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        _proc.kill()
    print("server stopped")
    _proc = None

## 6. Configs A + B — prefix cache **OFF**

In [ ]:
launch_vllm(prefix_cache=False)
wait_for_server()

In [ ]:
!python benchmark.py --config A
!python benchmark.py --config B

In [ ]:
stop_vllm()

## 7. Config C — prefix cache **ON** (restart server first)

In [ ]:
launch_vllm(prefix_cache=True)
wait_for_server()

In [ ]:
!python benchmark.py --config C

In [ ]:
stop_vllm()

## 8. Evaluate + plots

Computes F1 metrics and writes `results/evaluation.json` plus the latency-vs-length
and Pareto PDFs under `results/plots/`.

In [ ]:
!python evaluate.py

In [ ]:
import json, glob
print(json.dumps(json.load(open("results/evaluation.json")), indent=2))
print("\nplots:")
for p in sorted(glob.glob("results/plots/*.pdf")):
    print(" ", p)

## 9. Download results before disconnecting

`results/` is git-ignored and lives only on this VM. Grab the raw JSONL, the
aggregate/evaluation JSON, and the plot PDFs.

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob("results/**/*", recursive=True)):
    import os
    if os.path.isfile(f):
        files.download(f)